# FLIGHT DELAY: DATA CLEANING

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(
    '/kaggle/input/datasets/mostafa120/flights-sample-3m/flights_sample_3m.csv',
    low_memory=False
)

print(f"Raw shape: {df.shape}")
df.head(3)

Raw shape: (3000000, 32)


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,0.0,186.0,176.0,153.0,1065.0,NaN,NaN,NaN,NaN,NaN
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,0.0,235.0,236.0,189.0,1399.0,NaN,NaN,NaN,NaN,NaN
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,0.0,118.0,112.0,87.0,680.0,NaN,NaN,NaN,NaN,NaN


# Drop Completely Empty & Unnamed Columns

In [3]:
# Drop unnamed,empty columns created by Excel export
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

# Then I will drop rows where all core fields are null
core_cols = ['FL_DATE', 'AIRLINE', 'AIRLINE_CODE', 'ORIGIN', 'DEST']
df = df.dropna(subset=core_cols, how='all')

print(f"Shape after dropping empty rows and cols: {df.shape}")

Shape after dropping empty rows and cols: (3000000, 32)


# Fix Data Types

In [4]:
# I will parse the flight date properly
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'], infer_datetime_format=True, errors='coerce')

# extract useful time components
df['YEAR']    = df['FL_DATE'].dt.year
df['MONTH']   = df['FL_DATE'].dt.month
df['DAY']     = df['FL_DATE'].dt.day
df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek + 1  # 1=Monday 7=Sunday
df['MONTH_NAME']  = df['FL_DATE'].dt.strftime('%B')
df['DAY_NAME']    = df['FL_DATE'].dt.strftime('%A')

# convert to integer
int_cols = ['DOT_CODE', 'FL_NUMBER', 'CRS_DEP_TIME', 'CRS_ARR_TIME']
for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

print("Ya333 data types fixed")
df.dtypes

Ya333 data types fixed


FL_DATE                    datetime64[ns]
AIRLINE                            object
AIRLINE_DOT                        object
AIRLINE_CODE                       object
DOT_CODE                            Int64
FL_NUMBER                           Int64
ORIGIN                             object
ORIGIN_CITY                        object
DEST                               object
DEST_CITY                          object
CRS_DEP_TIME                        Int64
DEP_TIME                          float64
DEP_DELAY                         float64
TAXI_OUT                          float64
WHEELS_OFF                        float64
WHEELS_ON                         float64
TAXI_IN                           float64
CRS_ARR_TIME                        Int64
ARR_TIME                          float64
ARR_DELAY                         float64
CANCELLED                         float64
CANCELLATION_CODE                  object
DIVERTED                          float64
CRS_ELAPSED_TIME                  

# Clean & Standardize Text Columns

In [5]:
str_cols = ['AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'ORIGIN',
            'ORIGIN_CITY', 'DEST', 'DEST_CITY', 'CANCELLATION_CODE']

for col in str_cols:
    df[col] = df[col].astype(str).str.strip().str.upper()
    df[col] = df[col].replace('NAN', np.nan)

# Then I will extract a clean state from the city name  
df['ORIGIN_STATE'] = df['ORIGIN_CITY'].str.extract(r',\s*([A-Z]{2})$')
df['DEST_STATE']   = df['DEST_CITY'].str.extract(r',\s*([A-Z]{2})$')

# Clean city name by removing the state suffix
df['ORIGIN_CITY_CLEAN'] = df['ORIGIN_CITY'].str.replace(r',\s*[A-Z]{2}$', '', regex=True).str.strip()
df['DEST_CITY_CLEAN']   = df['DEST_CITY'].str.replace(r',\s*[A-Z]{2}$', '', regex=True).str.strip()

print("Ya333 text columns cleaned")

Ya333 text columns cleaned


# Handle Cancelled & Diverted Flights

In [6]:
# First, I will ensure CANCELLED and DIVERTED are clean binary integers
df['CANCELLED'] = pd.to_numeric(df['CANCELLED'], errors='coerce').fillna(0).astype(int)
df['DIVERTED']  = pd.to_numeric(df['DIVERTED'],  errors='coerce').fillna(0).astype(int)

# h7wl al cancellation codes ly readable labels
cancel_map = {
    'A': 'Carrier',
    'B': 'Weather',
    'C': 'NAS',
    'D': 'Security'
}
df['CANCELLATION_REASON'] = df['CANCELLATION_CODE'].map(cancel_map)

# For non cancelled flights, h7wl al delay columns from NaN to 0
delay_cols = ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER',
              'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']

# I will fill 0 where flight was NOT cancelled (cancelled flights have no delay data)
mask_not_cancelled = df['CANCELLED'] == 0
for col in delay_cols:
    df.loc[mask_not_cancelled, col] = df.loc[mask_not_cancelled, col].fillna(0)

print("Yes Cancellation & diversion columns handled Ya man")

Yes Cancellation & diversion columns handled Ya man


# Handle Missing Values in Numeric Columns

In [7]:
# For non-cancelled flights, I will fill the operational times with the median per route
operational_cols = ['DEP_DELAY', 'ARR_DELAY', 'TAXI_OUT', 'TAXI_IN',
                    'ELAPSED_TIME', 'AIR_TIME', 'CRS_ELAPSED_TIME']

for col in operational_cols:
    # I will fill in the missing values for non-cancelled flights using the overall median
    median_val = df.loc[mask_not_cancelled, col].median()
    df.loc[mask_not_cancelled, col] = df.loc[mask_not_cancelled, col].fillna(median_val)

print("missing numeric values ok")
print(df[operational_cols].isnull().sum())

missing numeric values ok
DEP_DELAY           77644
ARR_DELAY           79140
TAXI_OUT            78806
TAXI_IN             79140
ELAPSED_TIME        79140
AIR_TIME            79140
CRS_ELAPSED_TIME       14
dtype: int64


# Remove Duplicates

In [8]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate rows")
print(f"Final shape: {df.shape}")

Removed 0 duplicate rows
Final shape: (3000000, 43)


# Create Engineered Features (for Power BI deep analysis)

In [9]:
# 1: Delay classification
def classify_delay(minutes):
    if pd.isna(minutes) or minutes <= 0:
        return 'On Time - Early'
    elif minutes <= 15:
        return 'Minor (1-15 min)'
    elif minutes <= 45:
        return 'Moderate (16-45 min)'
    elif minutes <= 120:
        return 'Severe (46-120 min)'
    else:
        return 'Critical (>120 min)'

df['DEP_DELAY_CATEGORY'] = df['DEP_DELAY'].apply(classify_delay)
df['ARR_DELAY_CATEGORY']  = df['ARR_DELAY'].apply(classify_delay)

# 2: Is delayed flag, lazem ykon > 15 min
df['IS_DELAYED'] = ((df['ARR_DELAY'] > 15) & (df['CANCELLED'] == 0)).astype(int)

# 3: total delay breakdown label (dominant cause) 
def dominant_delay(row):
    causes = {
        'Carrier':       row.get('DELAY_DUE_CARRIER', 0) or 0,
        'Weather':       row.get('DELAY_DUE_WEATHER', 0) or 0,
        'NAS':           row.get('DELAY_DUE_NAS', 0) or 0,
        'Security':      row.get('DELAY_DUE_SECURITY', 0) or 0,
        'Late Aircraft': row.get('DELAY_DUE_LATE_AIRCRAFT', 0) or 0,
    }
    total = sum(causes.values())
    if total == 0:
        return 'No Delay'
    return max(causes, key=causes.get)

df['DOMINANT_DELAY_CAUSE'] = df.apply(dominant_delay, axis=1)

# 4: Time of day 
def time_of_day(ay_7aga):
    if pd.isna(ay_7aga):
        return np.nan
    h = int(ay_7aga) // 100
    if 5 <= h < 12:
        return 'Morning (5-11)'
    elif 12 <= h < 17:
        return 'Afternoon (12-4)'
    elif 17 <= h < 21:
        return 'Evening (5-8)'
    else:
        return 'Night (9-4)'

df['DEP_TIME_OF_DAY'] = df['CRS_DEP_TIME'].apply(time_of_day)

# 5: Distance 
def distance_band(d):
    if pd.isna(d): 
        return np.nan
    if d < 500:    
        return 'Short (<500 mi)'
    elif d < 1000: 
        return 'Medium (500-999 mi)'
    elif d < 2000: 
        return 'Long (1000-1999 mi)'
    else:          
        return 'Ultra-Long (≥2000 mi)'

df['DISTANCE_BAND'] = df['DISTANCE'].apply(distance_band)

# 6: Route key
df['ROUTE'] = df['ORIGIN'] + ' : ' + df['DEST']

print("Ya3333 engineered features created a5yran")
df.head(3)

Ya3333 engineered features created a5yran


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,ORIGIN_CITY_CLEAN,DEST_CITY_CLEAN,CANCELLATION_REASON,DEP_DELAY_CATEGORY,ARR_DELAY_CATEGORY,IS_DELAYED,DOMINANT_DELAY_CAUSE,DEP_TIME_OF_DAY,DISTANCE_BAND,ROUTE
0,2019-01-09,UNITED AIR LINES INC.,UNITED AIR LINES INC.: UA,UA,19977,1562,FLL,"FORT LAUDERDALE, FL",EWR,"NEWARK, NJ",...,FORT LAUDERDALE,NEWARK,NaN,On Time - Early,On Time - Early,0,No Delay,Morning (5-11),Long (1000-1999 mi),FLL : EWR
1,2022-11-19,DELTA AIR LINES INC.,DELTA AIR LINES INC.: DL,DL,19790,1149,MSP,"MINNEAPOLIS, MN",SEA,"SEATTLE, WA",...,MINNEAPOLIS,SEATTLE,NaN,On Time - Early,On Time - Early,0,No Delay,Night (9-4),Long (1000-1999 mi),MSP : SEA
2,2022-07-22,UNITED AIR LINES INC.,UNITED AIR LINES INC.: UA,UA,19977,459,DEN,"DENVER, CO",MSP,"MINNEAPOLIS, MN",...,DENVER,MINNEAPOLIS,NaN,Minor (1-15 min),On Time - Early,0,No Delay,Morning (5-11),Medium (500-999 mi),DEN : MSP


# Final Validation & Export

In [11]:
print("W da summary 3la 2d al7al kda ")
print(f"Total rows:      {len(df):,}")
print(f"Total columns:   {df.shape[1]}")
print(f"\nCancelled:       {df['CANCELLED'].sum():,}")
print(f"Diverted:        {df['DIVERTED'].sum():,}")
print(f"Delayed (>15m):  {df['IS_DELAYED'].sum():,}")
print(f"\nDate range: {df['FL_DATE'].min().date()} : {df['FL_DATE'].max().date()}")
print(f"\nAirlines: {df['AIRLINE_CODE'].nunique()}")
print(f"Airports:  {df['ORIGIN'].nunique()}")
print(f"\nRemaining nulls (key columns):")
key_check = ['FL_DATE','AIRLINE','ORIGIN','DEST','ARR_DELAY','CANCELLED','IS_DELAYED']
print(df[key_check].isnull().sum())

# h3ml export ll file 
df.to_csv('flights_cleaned.csv', index=False)
print("\n A333 DONE: flights_cleaned.csv exported, mubarak")

W da summary 3la 2d al7al kda 
Total rows:      3,000,000
Total columns:   50

Cancelled:       79,140
Diverted:        7,056
Delayed (>15m):  515,289

Date range: 2019-01-01 : 2023-08-31

Airlines: 18
Airports:  380

Remaining nulls (key columns):
FL_DATE           0
AIRLINE           0
ORIGIN            0
DEST              0
ARR_DELAY     79140
CANCELLED         0
IS_DELAYED        0
dtype: int64

 A333 DONE: flights_cleaned.csv exported, mubarak
